# Trabalho 9 — Redes Neurais: Perceptron de Uma Só Camada

**Disciplina:** Sistemas Inteligentes  

---

## 1. Introdução

O **Perceptron** é o modelo mais simples de rede neural artificial, proposto por Frank Rosenblatt em 1958. Ele consiste em um único neurônio que recebe múltiplas entradas, calcula uma soma ponderada e aplica uma **função de ativação** para produzir a saída.

### Modelo Matemático

Dado um vetor de entrada $\mathbf{x} = [x_1, x_2, \ldots, x_n]$ e um vetor de pesos $\mathbf{w} = [w_1, w_2, \ldots, w_n]$, a saída do perceptron é:

$$y = f\left(\sum_{i=1}^{n} w_i \cdot x_i + b\right)$$

onde $b$ é o **bias** (viés) e $f$ é a **função de ativação degrau** (step function):

$$f(z) = \begin{cases} 1, & \text{se } z \geq 0 \\ 0, & \text{se } z < 0 \end{cases}$$

### Regra de Aprendizado

Os pesos são atualizados pela regra:

$$w_i \leftarrow w_i + \eta \cdot (d - y) \cdot x_i$$
$$b \leftarrow b + \eta \cdot (d - y)$$

onde $\eta$ é a **taxa de aprendizado** e $d$ é a saída desejada.

## 2. Importação das Bibliotecas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Configurações de visualização
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12
plt.style.use('seaborn-v0_8-whitegrid')

## 3. Implementação do Perceptron

In [ ]:
class Perceptron:
    """
    Implementação do Perceptron de uma só camada.
    
    Parâmetros
    ----------
    taxa_aprendizado : float
        Taxa de aprendizado (eta), entre 0.0 e 1.0.
    n_epocas : int
        Número máximo de épocas (passagens pelo conjunto de treino).
    semente : int ou None
        Semente para reprodutibilidade dos pesos iniciais.
    
    Atributos
    ---------
    pesos_ : ndarray, shape (n_features,)
        Vetor de pesos após o treinamento.
    bias_ : float
        Bias (viés) após o treinamento.
    erros_por_epoca_ : list
        Número de classificações erradas em cada época.
    """
    
    def __init__(self, taxa_aprendizado=0.1, n_epocas=100, semente=None):
        self.taxa_aprendizado = taxa_aprendizado
        self.n_epocas = n_epocas
        self.semente = semente
    
    def _funcao_ativacao(self, z):
        """Função de ativação degrau (step function)."""
        return np.where(z >= 0, 1, 0)
    
    def _soma_ponderada(self, X):
        """Calcula a soma ponderada: z = X·w + b."""
        return np.dot(X, self.pesos_) + self.bias_
    
    def prever(self, X):
        """Realiza a predição para as entradas X."""
        return self._funcao_ativacao(self._soma_ponderada(X))
    
    def treinar(self, X, y):
        """
        Treina o perceptron com os dados de entrada X e rótulos y.
        
        Parâmetros
        ----------
        X : ndarray, shape (n_amostras, n_features)
            Dados de entrada.
        y : ndarray, shape (n_amostras,)
            Rótulos desejados (0 ou 1).
        
        Retorna
        -------
        self : objeto
        """
        # Inicialização dos pesos
        rng = np.random.default_rng(self.semente)
        self.pesos_ = rng.normal(loc=0.0, scale=0.01, size=X.shape[1])
        self.bias_ = 0.0
        self.erros_por_epoca_ = []
        
        print(f"Pesos iniciais: {self.pesos_}")
        print(f"Bias inicial:   {self.bias_}")
        print(f"Taxa de aprendizado: {self.taxa_aprendizado}")
        print(f"Épocas máximas: {self.n_epocas}")
        print("-" * 40)
        
        for epoca in range(1, self.n_epocas + 1):
            erros = 0
            
            for xi, yi in zip(X, y):
                # Predição
                y_pred = self.prever(xi)
                
                # Cálculo do erro
                erro = yi - y_pred
                
                # Atualização dos pesos e bias
                if erro != 0:
                    self.pesos_ += self.taxa_aprendizado * erro * xi
                    self.bias_ += self.taxa_aprendizado * erro
                    erros += 1
            
            self.erros_por_epoca_.append(erros)
            
            # Mostrar progresso a cada 10 épocas ou quando convergir
            if epoca % 10 == 0 or erros == 0:
                print(f"Época {epoca:3d} | Erros: {erros}")
            
            # Convergência: nenhum erro
            if erros == 0:
                print(f"\n✅ Convergiu na época {epoca}!")
                break
        else:
            print(f"\n⚠️ Não convergiu em {self.n_epocas} épocas.")
        
        print(f"\nPesos finais: {self.pesos_}")
        print(f"Bias final:   {self.bias_}")
        
        return self
    
    def acuracia(self, X, y):
        """Calcula a acurácia do modelo."""
        y_pred = self.prever(X)
        return np.mean(y_pred == y) * 100

## 4. Funções Auxiliares de Visualização

In [ ]:
def plotar_erros(perceptron, titulo="Erros por Época"):
    """Plota a curva de erros por época durante o treinamento."""
    plt.figure(figsize=(8, 4))
    epocas = range(1, len(perceptron.erros_por_epoca_) + 1)
    plt.plot(epocas, perceptron.erros_por_epoca_, marker='o', color='#e74c3c',
             linewidth=2, markersize=6)
    plt.xlabel('Época')
    plt.ylabel('Número de Erros')
    plt.title(titulo)
    plt.xticks(epocas)
    plt.ylim(bottom=-0.1)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plotar_fronteira_decisao(perceptron, X, y, titulo="Fronteira de Decisão"):
    """
    Plota os pontos de dados e a fronteira de decisão do perceptron.
    Funciona apenas para dados com 2 features.
    """
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Cores para as classes
    cores = ['#e74c3c', '#2ecc71']
    marcadores = ['o', 's']
    
    # Plotar os pontos
    for classe in np.unique(y):
        idx = y == classe
        ax.scatter(X[idx, 0], X[idx, 1],
                   c=cores[int(classe)], marker=marcadores[int(classe)],
                   s=150, edgecolors='black', linewidths=1.5,
                   label=f'Classe {int(classe)}', zorder=5)
    
    # Plotar a fronteira de decisão
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    
    # Região de decisão
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    Z = perceptron.prever(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.2, colors=cores, levels=[-0.5, 0.5, 1.5])
    ax.contour(xx, yy, Z, colors='black', linewidths=2, levels=[0.5])
    
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel('$x_1$', fontsize=14)
    ax.set_ylabel('$x_2$', fontsize=14)
    ax.set_title(titulo, fontsize=14)
    ax.legend(fontsize=12)
    plt.tight_layout()
    plt.show()


def tabela_resultados(X, y, y_pred, nomes_entradas=None):
    """Exibe uma tabela com entradas, saída desejada e saída do perceptron."""
    n_features = X.shape[1]
    if nomes_entradas is None:
        nomes_entradas = [f'x{i+1}' for i in range(n_features)]
    
    header = ' | '.join([f'{nome:>4}' for nome in nomes_entradas])
    header += ' | Desejado | Predito | Correto?'
    print(header)
    print('-' * len(header))
    
    for i in range(len(X)):
        entradas = ' | '.join([f'{X[i, j]:>4.0f}' for j in range(n_features)])
        correto = '✅' if y[i] == y_pred[i] else '❌'
        print(f'{entradas} |    {int(y[i]):>4}  |   {int(y_pred[i]):>4}  |   {correto}')

---

## 5. Experimento 1: Porta Lógica AND

A porta AND retorna 1 **somente** quando ambas as entradas são 1.

| $x_1$ | $x_2$ | $y$ |
|:-----:|:-----:|:---:|
| 0     | 0     | 0   |
| 0     | 1     | 0   |
| 1     | 0     | 0   |
| 1     | 1     | 1   |

Este é um problema **linearmente separável**, portanto o Perceptron deve convergir.

In [ ]:
# Dados da porta AND
X_and = np.array([[0, 0],
                  [0, 1],
                  [1, 0],
                  [1, 1]])

y_and = np.array([0, 0, 0, 1])

# Criar e treinar o perceptron
print("=" * 40)
print("   TREINAMENTO — PORTA AND")
print("=" * 40)

p_and = Perceptron(taxa_aprendizado=0.1, n_epocas=50, semente=42)
p_and.treinar(X_and, y_and)

In [ ]:
# Resultados
print("\n📊 Tabela de Resultados — AND")
print("=" * 40)
y_pred_and = p_and.prever(X_and)
tabela_resultados(X_and, y_and, y_pred_and)
print(f"\nAcurácia: {p_and.acuracia(X_and, y_and):.1f}%")

In [ ]:
# Visualizações
plotar_erros(p_and, titulo="Erros por Época — Porta AND")
plotar_fronteira_decisao(p_and, X_and, y_and, titulo="Fronteira de Decisão — Porta AND")

---

## 6. Experimento 2: Porta Lógica OR

A porta OR retorna 1 quando **pelo menos** uma das entradas é 1.

| $x_1$ | $x_2$ | $y$ |
|:-----:|:-----:|:---:|
| 0     | 0     | 0   |
| 0     | 1     | 1   |
| 1     | 0     | 1   |
| 1     | 1     | 1   |

Também é **linearmente separável**.

In [ ]:
# Dados da porta OR
X_or = np.array([[0, 0],
                 [0, 1],
                 [1, 0],
                 [1, 1]])

y_or = np.array([0, 1, 1, 1])

# Treinar
print("=" * 40)
print("   TREINAMENTO — PORTA OR")
print("=" * 40)

p_or = Perceptron(taxa_aprendizado=0.1, n_epocas=50, semente=42)
p_or.treinar(X_or, y_or)

In [ ]:
# Resultados
print("\n📊 Tabela de Resultados — OR")
print("=" * 40)
y_pred_or = p_or.prever(X_or)
tabela_resultados(X_or, y_or, y_pred_or)
print(f"\nAcurácia: {p_or.acuracia(X_or, y_or):.1f}%")

In [ ]:
# Visualizações
plotar_erros(p_or, titulo="Erros por Época — Porta OR")
plotar_fronteira_decisao(p_or, X_or, y_or, titulo="Fronteira de Decisão — Porta OR")

---

## 7. Experimento 3: Porta Lógica NAND

A porta NAND é o inverso da AND.

| $x_1$ | $x_2$ | $y$ |
|:-----:|:-----:|:---:|
| 0     | 0     | 1   |
| 0     | 1     | 1   |
| 1     | 0     | 1   |
| 1     | 1     | 0   |

Também é **linearmente separável**.

In [ ]:
# Dados da porta NAND
X_nand = np.array([[0, 0],
                   [0, 1],
                   [1, 0],
                   [1, 1]])

y_nand = np.array([1, 1, 1, 0])

# Treinar
print("=" * 40)
print("   TREINAMENTO — PORTA NAND")
print("=" * 40)

p_nand = Perceptron(taxa_aprendizado=0.1, n_epocas=50, semente=42)
p_nand.treinar(X_nand, y_nand)

In [ ]:
# Resultados
print("\n📊 Tabela de Resultados — NAND")
print("=" * 40)
y_pred_nand = p_nand.prever(X_nand)
tabela_resultados(X_nand, y_nand, y_pred_nand)
print(f"\nAcurácia: {p_nand.acuracia(X_nand, y_nand):.1f}%")

In [ ]:
# Visualizações
plotar_erros(p_nand, titulo="Erros por Época — Porta NAND")
plotar_fronteira_decisao(p_nand, X_nand, y_nand, titulo="Fronteira de Decisão — Porta NAND")

---

## 8. Experimento 4: Porta Lógica XOR (Limitação do Perceptron)

A porta XOR retorna 1 quando **exatamente** uma das entradas é 1.

| $x_1$ | $x_2$ | $y$ |
|:-----:|:-----:|:---:|
| 0     | 0     | 0   |
| 0     | 1     | 1   |
| 1     | 0     | 1   |
| 1     | 1     | 0   |

⚠️ **Este problema NÃO é linearmente separável!** O Perceptron de uma só camada **não** consegue resolver o XOR — esta é a limitação clássica demonstrada por Minsky & Papert (1969).

In [ ]:
# Dados da porta XOR
X_xor = np.array([[0, 0],
                  [0, 1],
                  [1, 0],
                  [1, 1]])

y_xor = np.array([0, 1, 1, 0])

# Treinar
print("=" * 40)
print("   TREINAMENTO — PORTA XOR")
print("=" * 40)

p_xor = Perceptron(taxa_aprendizado=0.1, n_epocas=50, semente=42)
p_xor.treinar(X_xor, y_xor)

In [ ]:
# Resultados
print("\n📊 Tabela de Resultados — XOR")
print("=" * 40)
y_pred_xor = p_xor.prever(X_xor)
tabela_resultados(X_xor, y_xor, y_pred_xor)
print(f"\nAcurácia: {p_xor.acuracia(X_xor, y_xor):.1f}%")

In [ ]:
# Visualizações
plotar_erros(p_xor, titulo="Erros por Época — Porta XOR (não converge!)")
plotar_fronteira_decisao(p_xor, X_xor, y_xor, titulo="Fronteira de Decisão — Porta XOR (falha!)")

---

## 9. Experimento 5: Classificação com Dataset Iris (2 classes)

Agora vamos testar o Perceptron em um problema real: classificar flores do dataset **Iris** (apenas 2 classes: Setosa vs. Versicolor) usando 2 atributos (comprimento e largura da sépala).

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Carregar o dataset Iris
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Selecionar apenas Setosa (0) e Versicolor (1)
# Usar apenas 2 features: comprimento e largura da sépala
mask = y_iris < 2
X_iris_2 = X_iris[mask, :2]  # sepal length e sepal width
y_iris_2 = y_iris[mask]

print(f"Amostras: {X_iris_2.shape[0]}")
print(f"Features: {iris.feature_names[:2]}")
print(f"Classes:  {iris.target_names[:2]}")
print(f"\nDistribuição das classes:")
print(f"  Setosa:     {np.sum(y_iris_2 == 0)}")
print(f"  Versicolor: {np.sum(y_iris_2 == 1)}")

In [ ]:
# Dividir em treino e teste
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X_iris_2, y_iris_2, test_size=0.3, random_state=42, stratify=y_iris_2
)

# Normalização (padronização z-score)
scaler = StandardScaler()
X_treino_norm = scaler.fit_transform(X_treino)
X_teste_norm = scaler.transform(X_teste)

print(f"Amostras de treino: {X_treino_norm.shape[0]}")
print(f"Amostras de teste:  {X_teste_norm.shape[0]}")

In [ ]:
# Treinar
print("=" * 40)
print("   TREINAMENTO — IRIS (Setosa vs Versicolor)")
print("=" * 40)

p_iris = Perceptron(taxa_aprendizado=0.01, n_epocas=100, semente=42)
p_iris.treinar(X_treino_norm, y_treino)

In [ ]:
# Resultados
acc_treino = p_iris.acuracia(X_treino_norm, y_treino)
acc_teste = p_iris.acuracia(X_teste_norm, y_teste)

print(f"\n📊 Resultados — Iris")
print("=" * 40)
print(f"Acurácia no Treino: {acc_treino:.1f}%")
print(f"Acurácia no Teste:  {acc_teste:.1f}%")

In [ ]:
# Visualizações
plotar_erros(p_iris, titulo="Erros por Época — Iris (Setosa vs Versicolor)")
plotar_fronteira_decisao(p_iris, X_treino_norm, y_treino,
                         titulo="Fronteira de Decisão — Iris (Treino)")
plotar_fronteira_decisao(p_iris, X_teste_norm, y_teste,
                         titulo="Fronteira de Decisão — Iris (Teste)")

---

## 10. Comparativo: Influência da Taxa de Aprendizado

Vamos analisar como diferentes taxas de aprendizado afetam a convergência do Perceptron na porta AND.

In [ ]:
taxas = [0.01, 0.1, 0.5, 1.0]
resultados = {}

fig, axes = plt.subplots(1, len(taxas), figsize=(16, 4), sharey=True)

for i, eta in enumerate(taxas):
    p = Perceptron(taxa_aprendizado=eta, n_epocas=30, semente=42)
    p.treinar(X_and, y_and)
    
    epocas = range(1, len(p.erros_por_epoca_) + 1)
    axes[i].plot(epocas, p.erros_por_epoca_, marker='o', color='#3498db',
                 linewidth=2, markersize=5)
    axes[i].set_title(f'η = {eta}', fontsize=13)
    axes[i].set_xlabel('Época')
    if i == 0:
        axes[i].set_ylabel('Erros')
    axes[i].grid(True, alpha=0.3)
    
    convergiu = p.erros_por_epoca_[-1] == 0
    epoca_conv = len(p.erros_por_epoca_) if convergiu else '-'
    resultados[eta] = {'convergiu': convergiu, 'epoca': epoca_conv}

plt.suptitle('Influência da Taxa de Aprendizado — Porta AND', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Tabela resumo
print("\n📊 Resumo — Taxa de Aprendizado")
print("=" * 40)
print(f"{'η':>6} | {'Convergiu?':>12} | {'Época':>6}")
print("-" * 30)
for eta, res in resultados.items():
    conv_str = '✅ Sim' if res['convergiu'] else '❌ Não'
    print(f"{eta:>6} | {conv_str:>12} | {res['epoca']:>6}")

---

## 11. Conclusões

### Resultados Obtidos

| Problema | Linearmente Separável? | Perceptron Convergiu? |
|----------|:---------------------:|:--------------------:|
| AND      | ✅ Sim                | ✅ Sim               |
| OR       | ✅ Sim                | ✅ Sim               |
| NAND     | ✅ Sim                | ✅ Sim               |
| XOR      | ❌ Não                | ❌ Não               |
| Iris (2 classes) | ✅ Sim         | ✅ Sim               |

### Principais Observações

1. **Teorema de Convergência do Perceptron**: O Perceptron sempre converge para problemas **linearmente separáveis**, independente da inicialização dos pesos.

2. **Limitação do XOR**: O Perceptron de uma só camada não consegue resolver o problema XOR porque as classes não são linearmente separáveis — é impossível traçar uma única reta que separe as duas classes. Esta foi a principal crítica de Minsky & Papert (1969).

3. **Taxa de Aprendizado**: Taxas maiores podem levar a uma convergência mais rápida (menos épocas), mas em problemas mais complexos podem causar instabilidade.

4. **Normalização**: Para dados reais (como o Iris), a normalização dos dados é importante para o bom funcionamento do Perceptron.

### Solução para o XOR

Para resolver problemas não linearmente separáveis como o XOR, é necessário usar redes com **múltiplas camadas** (Multilayer Perceptron — MLP), que introduzem camadas ocultas e funções de ativação não lineares.